# LINEAR-01 — Espaces vectoriels et dimension finie : les marches d'Axler face a Mathlib

Sous-serie **LINEAR** (11e sous-serie, schema canonique #17545) - Fille 3.2 de l'Epic Origami #24838 :
ce carnet transpose en prose pedagogique la **marche 1** du lac `linear_lean` (chapitres 1-2 d'Axler),
ecrite en `.lean` dans la PR #26894. Chaque enonce est verifie ici **dans le kernel Lean** : le carnet
n'affirme rien que le type-checker ne confirme cellule par cellule.

La sous-serie suit l'ordre d'Axler (*Linear Algebra Done Right*, 4e ed.) : espaces et dimension ici,
applications lineaires en LINEAR-02, puis polynomes, valeurs propres, produits scalaires, operateurs,
formes canoniques, et la trace et le determinant en dernier (LINEAR-09) - le choix assumé d'Axler.

**Navigation** : [Serie Lean](../README.md) - [Sous-serie LINEAR](README.md) - **LINEAR-01** -
[LINEAR-02 Applications lineaires ->](LINEAR-02-Applications-Lineaires-Lean.ipynb)

## Sources et licence

- **Source unique du fond mathematique** : Sheldon Axler, *Linear Algebra Done Right*, 4e ed., Springer 2024,
  **CC BY-NC 4.0** (version du 30 octobre 2026). Copie archivee au gisement partage
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Pedagogie-Proves\` (regle `bibliography-hygiene` : chaque
  enonce est ecrit depuis la copie archivee, jamais de memoire).
- **Aucune ligne recopiee des compagnons** : `rkirov/linear-algebra-done-right-lean` (Apache-2.0, attribution
  dans chaque carnet consommateur a partir de LINEAR-03) et `blanketism/Axler-Solutions` (sans licence
  declaree) ont ete consultes uniquement pour comparer les progressions, jamais vendus. Les definitions
  sont portees par les organes natifs de Mathlib, declares et cites.
- Le lac distille **l'ordre** d'Axler, pas son texte : `Submodule` (Mathlib) joue les sous-espaces,
  `Basis` les bases, `finrank` la dimension.

In [1]:
import Linear.EspacesVectoriels
import Linear.DimensionFinie

open Linear

import Linear.EspacesVectoriels
import Linear.DimensionFinie

open Linear
--% env 0

Raw input:
{"cmd": "import Linear.EspacesVectoriels\nimport Linear.DimensionFinie\n\nopen Linear"}
Raw output:
{"env": 0}

## 0. Statut epistemique

Tous les theoremes du texte (Axler 1.31 a 2.43) sont **notre** marche 1 : prouves dans le lac `linear_lean`,
sur les structures natives de Mathlib (`Field`, `AddCommGroup`, `Module`, `Submodule`, `Basis`, `finrank`),
avec zero `sorry` et zero `native_decide` - l'empreinte exacte des axiomes est affichee en fin de carnet.
Les **exercices** d'Axler relevent d'un statut distinct : quatre sorry-cibles documentees (mandat user
2026-11-03) - enonces charges sous numero Axler, preuves laissees au prouveur, comptees dans la baseline
CI du lac (4). Ce carnet affiche l'empreinte honnete des deux : axiomes standards sur les theoremes,
`sorryAx` sur la cible 1B.3.

## 1. Chapitre 1 : l'algebre des scalaires et des sous-espaces

Le premier chapitre d'Axler installe la mecanique : que font zero et -1 sur un vecteur, comment se
comportent les sommes de sous-espaces, et quand une somme est-elle directe. Notre marche encode ces
gestes en quatre theoremes et trois exercices.

### 1.1 Scalaires et vecteur nul (Axler 1.31-1.32)

Dans une algebre lineaire sur un corps, deux identites paraissent evidentes et se demontrent en une
ligne chacune : multiplier le vecteur nul par un scalaire donne le vecteur nul, et multiplier par -1
donne l'oppose. Mathlib les attend sur la structure `Module K V` la plus generale.

In [2]:
#check scalaire_fois_zero
#check moins_un_fois_vecteur

#check scalaire_fois_zero
──────▶  Linear.scalaire_fois_zero.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (a : K) :
  a • 0 = 0
#check moins_un_fois_vecteur
──────▶  Linear.moins_un_fois_vecteur.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (v : V) :
  -1 • v = -v
--% env 1

Raw input:
{"cmd": "#check scalaire_fois_zero\n#check moins_un_fois_vecteur", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.scalaire_fois_zero.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (a : K) :\n  a • 0 = 0"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.moins_un_fois_vecteur.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (v : V) :\n  -1 • v = -v"}],
 "env": 1}

### Lecture du resultat

`scalaire_fois_zero (a : K) : a • (0 : V) = 0` et `moins_un_fois_vecteur (v : V) : (-1 : K) • v = -v` :
les hypotheses de type (`[Field K] [AddCommGroup V] [Module K V]`) sont exactement la structure de
module sur un corps - ni plus, ni moins. Chacune delegue a son organe Mathlib (`smul_zero`,
`neg_one_smul`) : la marche n'ecrit que l'enonce sous son numero d'Axler. C'est la premiere lecon de la
marche : **ecrire l'enonce minimal, deleguer la preuve a l'organe**.

### 1.2 Sommes de sous-espaces (Axler 1.44-1.45)

La somme `U + W` de deux sous-espaces est le plus petit sous-espace les contenant. Deux questions
naturelles : qui appartient a la somme, et quand la decomposition est-elle unique ? La reponse a la
seconde definit la somme directe.

In [3]:
#check mem_somme
#check decomposition_unique_somme_directe

#check mem_somme
──────▶  Linear.mem_somme.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (U W : Submodule K V)
  (x : V) : x ∈ U ⊔ W ↔ ∃ u ∈ U, ∃ w ∈ W, x = u + w
#check decomposition_unique_somme_directe
──────▶  Linear.decomposition_unique_somme_directe.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V]
  [Module K V] {U W : Submodule K V} (h : IsCompl U W) (v : V) : ∃! p, ↑p.1 + ↑p.2 = v
--% env 2

Raw input:
{"cmd": "#check mem_somme\n#check decomposition_unique_somme_directe", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.mem_somme.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (U W : Submodule K V)\n  (x : V) : x ∈ U ⊔ W ↔ ∃ u ∈ U, ∃ w ∈ W, x = u + w"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.decomposition_unique_somme_directe.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V]\n  [Module K V] {U W : Submodule K V} (h : IsCompl U W) (v : V) : ∃! p, ↑p.1 + ↑p.2 = v"}],
 "env": 2}

### Lecture du resultat

`mem_somme` dit que `x ∈ U ⊔ W` ssi `x` s'ecrit `u + w` avec `u ∈ U`, `w ∈ W` - c'est le pont entre
l'existence d'une decomposition et l'appartenance (organe : `Submodule.mem_sup`).
`decomposition_unique_somme_directe` prend la somme directe **donnee** (`h : IsCompl U W` - le critere
1.45 `U ⊓ W = ⊥` est porte par la structure Mathlib) et produit le `∃!` : chaque vecteur de `V` a
exactement une paire `(u, w)`. C'est l'une des rares preuves ecrites a la main dans la marche (de
`mem_sup` jusqu'a `inf_eq_bot`) ; l'essentiel du reste du lac delegue.

### 1.3 Les exercices 1B d'Axler : deux echauffements resolus, une sorry-cible

Le chapitre 1B accompagne la marche de trois exercices. Les deux premiers sont des echauffements que
l'organe Mathlib disperse en une ligne - la marche les montre resolus pour exhiber le geste. Le
troisieme est une **sorry-cible documentee** : l'enonce est charge, la preuve est laisse au prouveur.

1. **1B.1** (resolu) - l'oppose de l'oppose : `-(-v) = v`, organe `neg_neg`.
2. **1B.2** (resolu) - annulation : si `a • v = 0` alors `a = 0` ou `v = 0`, organe `smul_eq_zero`.
3. **1B.3** (sorry-cible) - existence et unicite de la solution de `v + 3 • x = w`, sous `[CharZero K]`.

In [4]:
#check exercice_1B_1
#check exercice_1B_2
#check exercice_1B_3

#check exercice_1B_1
──────▶  Linear.exercice_1B_1.{u_2} {V : Type u_2} [AddCommGroup V] (v : V) : - -v = v
#check exercice_1B_2
──────▶  Linear.exercice_1B_2.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (a : K) (v : V)
  (h : a • v = 0) : a = 0 ∨ v = 0
#check exercice_1B_3
──────▶  Linear.exercice_1B_3.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] [CharZero K]
  (v w : V) : ∃! x, v + 3 • x = w
--% env 3

Raw input:
{"cmd": "#check exercice_1B_1\n#check exercice_1B_2\n#check exercice_1B_3", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.exercice_1B_1.{u_2} {V : Type u_2} [AddCommGroup V] (v : V) : - -v = v"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.exercice_1B_2.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] (a : K) (v : V)\n  (h : a • v = 0) : a = 0 ∨ v = 0"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Linear.exercice_1B_3.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] [CharZero K]\n  (v w : V) : ∃! x, v + 3 • x = w"}],
 "env": 3}

### Lecture des trois exercices

1B.1 est l'involution de l'oppose ; 1B.2 utilise que `K` est un `Field` (pas d'anneau a diviseurs de
zero). 1B.3 produit un `∃!` qui prepare le critere de somme directe de la section 1.2 ; son hypothese
`[CharZero K]` est un correctif de la review de la PR #26894 : en caracteristique 3, `(3 : K) = 0` et
l'enonce devient faux (`v + 3 • x = v` pour tout `x`, l'unicite exigerait `v = w`) ; `CharZero` rend
`3` inversible, fidele au cadre d'Axler (R ou C). La preuve reste a ecrire - cible prover `easy` :
construire `x = (3 : K)⁻¹ • (w - v)`, puis l'unicite par difference. Le `#check` confirme l'enonce ;
il ne certifie pas la preuve.

## 2. Chapitre 2 : la dimension comme invariant

Le chapitre 2 d'Axler remplace le calcul par un invariant : la **dimension**. Notre marche en donne
deux volets - la theorie des familles (libres, couvrantes, bases), puis les theoremes de la bonne
taille, dont Grassmann est le pivot.

### 2.1 Familles libres et lemme de dependance (Axler 2.19, 2.22)

Une famille libre ne contient pas de combinaison non triviale qui s'annule ; une famille couvrante
engendre tout l'espace. Le lemme de dependance 2.19 dit qu'une famille finie dependante admet une
combinaison non triviale nulle ; 2.22 en tire qu'une famille libre n'est jamais plus longue qu'une
famille couvrante - c'est ce qui rend la dimension bien definie.

In [5]:
#check combinaison_non_triviale
#check longueur_libre_le_couverte

#check combinaison_non_triviale
──────▶  Linear.combinaison_non_triviale.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] {n : ℕ}
  (v : Fin n → V) (h : ¬LinearIndependent K v) : ∃ g, ∑ i, g i • v i = 0 ∧ ∃ i, g i ≠ 0
#check longueur_libre_le_couverte
──────▶  Linear.longueur_libre_le_couverte.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  [Module.Finite K V] {m n : ℕ} (v : Fin m → V) (hv : LinearIndependent K v) (w : Fin n → V)
  (hw : Submodule.span K (Set.range w) = ⊤) : m ≤ n
--% env 4

Raw input:
{"cmd": "#check combinaison_non_triviale\n#check longueur_libre_le_couverte", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.combinaison_non_triviale.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] {n : ℕ}\n  (v : Fin n → V) (h : ¬LinearIndependent K v) : ∃ g, ∑ i, g i • v i = 0 ∧ ∃ i, g i ≠ 0"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.longueur_libre_le_couverte.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  [Module.Finite K V] {m n : ℕ} (v : Fin m → V) (hv : LinearIndependent K v) (w : Fin n → V)\n  (hw : Submodule.span K (Set.range w) = ⊤) : m ≤ n"}],
 "env": 4}

### Lecture du resultat

`combinaison_non_triviale` : une famille finie **dependante** admet une combinaison lineaire non
triviale egale a zero - la forme existence du lemme 2.19 (l'indice minimal du livre est la technique
de preuve ; l'organe est `Fintype.linearIndependent_iff`). `longueur_libre_le_couverte` en tire
`m ≤ n` : la preuve est le premier calcul de dimension de la marche - `finrank_span_eq_card`
(l'envergement d'une famille libre de `m` vecteurs a dimension `m`), puis
`finrank_le_of_span_eq_top`, puis `omega`. Tout le reste du chapitre en est consequence formelle.

### 2.2 Bases : critere, complement, dimension (Axler 2.28, 2.33, 2.35)

Trois visages de la meme notion : une famille qui est libre ET couvrante est une base ; tout
sous-espace admet un complement ; et la longueur d'une base ne depend pas de la base choisie -
c'est la dimension.

In [6]:
#check critere_base
#check complement_existe
#check dimension_base

#check critere_base
──────▶  Linear.critere_base.{u_1, u_2, u_3} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] {ι : Type u_3}
  [Fintype ι] (v : ι → V) (x : V) : x ∈ Submodule.span K (Set.range v) ↔ ∃ c, ∑ i, c i • v i = x
#check complement_existe
──────▶  Linear.complement_existe.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  (U : Submodule K V) : ∃ W, IsCompl U W
#check dimension_base
──────▶  Linear.dimension_base.{u_1, u_2, u_3} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  {ι : Type u_3} [Fintype ι] (b : Module.Basis ι K V) : Module.finrank K V = Fintype.card ι
--% env 5

Raw input:
{"cmd": "#check critere_base\n#check complement_existe\n#check dimension_base", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.critere_base.{u_1, u_2, u_3} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] {ι : Type u_3}\n  [Fintype ι] (v : ι → V) (x : V) : x ∈ Submodule.span K (Set.range v) ↔ ∃ c, ∑ i, c i • v i = x"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.complement_existe.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  (U : Submodule K V) : ∃ W, IsCompl U W"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Linear.dimension_base.{u_1, u_2, u_3} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  {ι : Type u_3} [Fintype ι] (b : Module.Basis ι K V) : Module.finrank K V = Fintype.card ι"}],
 "env": 5}

### Lecture du resultat

`critere_base` : `x` appartient a l'envergement de la famille `v` si et seulement si `x` est
combinaison lineaire des `v i` - la lecture coefficients de la definition 2.26, portee par
`mem_span_range_iff_exists_fun`. `complement_existe` : tout sous-espace admet un supplementaire,
delegue a `Submodule.exists_isCompl` - sans meme exiger la dimension finie. `dimension_base` fixe
la definition : `finrank K V = Fintype.card ι` pour toute base `b : Basis ι K V` - le fait que
toutes les bases ont meme longueur (2.36) est porte par l'organe `finrank_eq_card_basis`.

### 2.3 La bonne taille suffit (Axler 2.37-2.42)

Dans un espace de dimension `n`, une famille libre de `n` vecteurs est une base ; une famille
couvrante de `n` vecteurs est une base ; un sous-espace de dimension `n` est tout l'espace. La
dimension ne pardonne pas : la bonne taille transforme une moitie d'hypothese en conclusion entiere.

In [7]:
#check famille_libre_bonne_taille_est_base
#check famille_couvante_bonne_taille_est_base
#check sous_espace_pleine_dimension
#check dim_sous_espace_le

#check famille_libre_bonne_taille_est_base
──────▶  Linear.famille_libre_bonne_taille_est_base.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V]
  [Module K V] [Module.Finite K V] {n : ℕ} (v : Fin n → V) (hv : LinearIndependent K v)
  (hcard : Module.finrank K V = n) : Submodule.span K (Set.range v) = ⊤
#check famille_couvante_bonne_taille_est_base
──────▶  Linear.famille_couvante_bonne_taille_est_base.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V]
  [Module K V] [Module.Finite K V] {n : ℕ} (v : Fin n → V) (hspan : Submodule.span K (Set.range v) = ⊤)
  (hcard : Module.finrank K V = n) : LinearIndependent K v
#check sous_espace_pleine_dimension
──────▶  Linear.sous_espace_pleine_dimension.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  [Module.Finite K V] (U : Submodule K V) (h : Module.finrank K ↥U = Module.finrank K V) : U = ⊤
#check dim_sous_espace_le
──────▶  Linear.dim_sous_espace_le.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  [Module.Finite K V] (U : Submodule K V) : Module.finrank K ↥U ≤ Module.finrank K V
--% env 6

Raw input:
{"cmd": "#check famille_libre_bonne_taille_est_base\n#check famille_couvante_bonne_taille_est_base\n#check sous_espace_pleine_dimension\n#check dim_sous_espace_le", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.famille_libre_bonne_taille_est_base.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V]\n  [Module K V] [Module.Finite K V] {n : ℕ} (v : Fin n → V) (hv : LinearIndependent K v)\n  (hcard : Module.finrank K V = n) : Submodule.span K (Set.range v) = ⊤"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.famille_couvante_bonne_taille_est_base.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V]\n  [Module K V] [Module.Finite K V] {n : ℕ} (v : Fin n → V) (hspan : Submodule.span K (Set.range v) = ⊤)\n  (hcard : Module.finrank K V = n) : LinearIndependent K v"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Linear.sous_espace_pleine_dimension.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  [Module.Finite K V] (U : Submodule K V) (h : Module.finrank K ↥U = Module.finrank K V) : U = ⊤"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Linear.dim_sous_espace_le.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  [Module.Finite K V] (U : Submodule K V) : Module.finrank K ↥U ≤ Module.finrank K V"}],
 "env": 6}

### Lecture du resultat

Les quatre enonces forment un carre. `dim_sous_espace_le` donne l'inegalite (2.37) ; les deux
criteres de bonne taille tournent l'inegalite en conclusion entiere par chaque cote : libre de `n`
vecteurs, l'envergement est ⊤ (2.38, par `eq_top_of_finrank_eq`) ; couvrante de `n` vecteurs, la
famille est libre (2.42) ; `sous_espace_pleine_dimension` ferme le carre (2.39). Une exception de
methode a noter : 2.42 est prouve par l'argument d'application lineaire (rang + noyau) qui annonce
le chapitre 3 - signale dans sa docstring, c'est le pont vers LINEAR-02.

### 2.4 Grassmann (Axler 2.43) - le pivot de la marche 1

`dim (U ∩ W) + dim (U + W) = dim U + dim W` : la formule de Grassmann est le theoreme-pivot du
chapitre 2. Elle relie l'intersection a la somme, et prepares les calculs de dimension des
exercices 2C comme le theoreme du rang de LINEAR-02.

In [8]:
#check grassmann

#check grassmann
──────▶  Linear.grassmann.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] [Module.Finite K V]
  (U W : Submodule K V) :
  Module.finrank K ↥(U ⊔ W) + Module.finrank K ↥(U ⊓ W) = Module.finrank K ↥U + Module.finrank K ↥W
--% env 7

Raw input:
{"cmd": "#check grassmann", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.grassmann.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] [Module.Finite K V]\n  (U W : Submodule K V) :\n  Module.finrank K ↥(U ⊔ W) + Module.finrank K ↥(U ⊓ W) = Module.finrank K ↥U + Module.finrank K ↥W"}],
 "env": 7}

### Lecture du resultat

`grassmann [Module.Finite K V] (U W : Submodule K V) :
finrank K ↥(U ⊔ W) + finrank K ↥(U ⊓ W) = finrank K ↥U + finrank K ↥W` - la forme additive de la
formule 2.43 (Axler l'ecrit `dim (U + W) = dim U + dim W − dim (U ∩ W)`). La marche delegue a
l'organe `Submodule.finrank_sup_add_finrank_inf_eq` : la preuve complete vit dans Mathlib, la marche
n'en porte que le numero d'Axler et l'ordre.

### 2.5 Les exercices 2A et 2C d'Axler : trois sorry-cibles documentees

Trois exercices accompagnent le chapitre 2. Tous trois sont des **sorry-cibles** (mandat user
2026-11-03) : l'enonce est charge sous numero Axler, la preuve est laisse au prouveur, et la cible
est documentee dans la docstring (`Cible prover : <niveau> - <tentative>`). Le `#check` confirme
l'enonce ; il ne certifie pas la preuve.

1. **2A.13** - si `v` est libre et que la famille translatee `v i + w` est dependante, alors `w`
   est dans l'envergement de `v`.
2. **2C.12** - en dimension 8, des sous-espaces de dimensions 3 et 5 dont la somme est l'espace
   entier sont supplementaires (`IsCompl`).
3. **2C.13** - en dimension 9, deux sous-espaces de dimension 5 ont une intersection non nulle.

In [9]:
#check exercice_2A_13
#check exercice_2C_12
#check exercice_2C_13

#check exercice_2A_13
──────▶  Linear.exercice_2A_13.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] {n : ℕ}
  (v : Fin n → V) (w : V) (hv : LinearIndependent K v) (h : ¬LinearIndependent K fun i => v i + w) :
  w ∈ Submodule.span K (Set.range v)
#check exercice_2C_12
──────▶  Linear.exercice_2C_12.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  [Module.Finite K V] (U W : Submodule K V) (hdim : Module.finrank K V = 8) (hU : Module.finrank K ↥U = 3)
  (hW : Module.finrank K ↥W = 5) (hsup : U ⊔ W = ⊤) : IsCompl U W
#check exercice_2C_13
──────▶  Linear.exercice_2C_13.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]
  [Module.Finite K V] (U W : Submodule K V) (hdim : Module.finrank K V = 9) (hU : Module.finrank K ↥U = 5)
  (hW : Module.finrank K ↥W = 5) : U ⊓ W ≠ ⊥
--% env 8

Raw input:
{"cmd": "#check exercice_2A_13\n#check exercice_2C_12\n#check exercice_2C_13", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Linear.exercice_2A_13.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V] {n : ℕ}\n  (v : Fin n → V) (w : V) (hv : LinearIndependent K v) (h : ¬LinearIndependent K fun i => v i + w) :\n  w ∈ Submodule.span K (Set.range v)"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Linear.exercice_2C_12.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  [Module.Finite K V] (U W : Submodule K V) (hdim : Module.finrank K V = 8) (hU : Module.finrank K ↥U = 3)\n  (hW : Module.finrank K ↥W = 5) (hsup : U ⊔ W = ⊤) : IsCompl U W"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Linear.exercice_2C_13.{u_1, u_2} {K : Type u_1} {V : Type u_2} [Field K] [AddCommGroup V] [Module K V]\n  [Module.Finite K V] (U W : Submodule K V) (hdim : Module.finrank K V = 9) (hU : Module.finrank K ↥U = 5)\n  (hW : Module.finrank K ↥W = 5) : U ⊓ W ≠ ⊥"}],
 "env": 8}

### Lecture des trois cibles

2A.13 (cible medium) : contraposee sur une relation de dependance `∑ g i • (v i + w) = 0`,
distinguer `∑ g i = 0` (donne une dependance sur `v`, contradiction) de `∑ g i ≠ 0` (isole `w`).
2C.12 (easy-medium) : Grassmann donne `finrank (U ⊓ W) = 3 + 5 − 8 = 0`, donc `U ⊓ W = ⊥`, reste a
conclure `IsCompl`. 2C.13 (easy-medium) : par l'absurde, `U ⊓ W = ⊥` forcerait
`finrank (U ⊔ W) = 10 > 9`. Ces trois enonces sont comptes dans la baseline CI du lac (4 avec 1B.3) :
les prouver est un gain mesurable, pas une surprise.

## 3. Empreinte des axiomes et conclusion de la marche 1

Les theoremes du texte reposent sur les axiomes standards de Mathlib (`propext`, `Classical.choice`,
`Quot.sound`). A cote, l'empreinte de la sorry-cible 1B.3 : `sorryAx` - la signature honnete d'une
preuve non ecrite, celle que la baseline CI du lac compte :

In [10]:
#print axioms grassmann
#print axioms decomposition_unique_somme_directe
#print axioms exercice_1B_3

#print axioms grassmann
──────▶  'Linear.grassmann' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms decomposition_unique_somme_directe
──────▶  'Linear.decomposition_unique_somme_directe' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms exercice_1B_3
──────▶  'Linear.exercice_1B_3' depends on axioms: [propext, sorryAx, Quot.sound]
--% env 9

Raw input:
{"cmd": "#print axioms grassmann\n#print axioms decomposition_unique_somme_directe\n#print axioms exercice_1B_3", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Linear.grassmann' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'Linear.decomposition_unique_somme_directe' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'Linear.exercice_1B_3' depends on axioms: [propext, sorryAx, Quot.sound]"}],
 "env": 9}

### Ce que la marche 1 installe pour la suite

Avec le chapitre 2, la dimension est un invariant utilisable : la marche 2 (LINEAR-02) y greffera
les applications lineaires - noyau, image, theoreme du rang - et la marche 3 les polynomes. Le
capstone `Lean-11c` refermera l'escalier en demonstrant un resultat charniere qui suppose
exactement ces deux marches.

**Navigation** : [LINEAR-02 Applications lineaires ->](LINEAR-02-Applications-Lineaires-Lean.ipynb) -
[Sous-serie LINEAR](README.md) - [Serie Lean](../README.md)